# 09 · Hệ thống đầu–cuối: gói mô hình → API → CSDL (phụ trách: Chưởng)

**Đề:** *"Experimental prototype system … supporting mortality/readmission/outcome prediction with explainability, calibration, subgroup analysis, and drift monitoring"*.
Mã nguồn: `trustworthy_ai/bundle.py`, `backend/app/`, `deploy/docker-compose.yml`.

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

In [ ]:
os.environ.setdefault("DATABASE_URL", "sqlite:///artifacts/notebook_demo.db")
from fastapi.testclient import TestClient
from backend.app.main import app
c = TestClient(app); c.__enter__()
pd.DataFrame(c.get("/api/v1/tasks").json())[["task", "name", "enabled", "model_version"]]

In [ ]:
rows = []
for t in ("p1", "p2", "p3"):
    for case in c.get(f"/api/v1/cases/{t}").json():
        r = c.post(f"/api/v1/predict/{t}", json={"case_id": case["case_id"], "features": case["features"]}).json()
        rows.append({"task": t, "ca": case["title"], "xác suất": r["calibrated_probability"], "mức": r["risk_level"],
                     "review": r["review"]["flag"], "lý do": ", ".join(r["review"]["reasons"])[:80]})
pd.DataFrame(rows)

In [ ]:
c.get('/api/v1/admin/system').json()

In [ ]:
pd.DataFrame(c.get('/api/v1/admin/models').json())[['task', 'version', 'model', 'calibration', 'cv_pr_auc', 'test_pr_auc', 'serving']]

## Câu hỏi tự kiểm tra
1. Gói mô hình gồm những file nào? SHA256SUMS dùng để làm gì?
2. Vì sao API không bao giờ huấn luyện lại mô hình?
3. Một dự đoán trả về những trường nào để đáp ứng "trustworthy"?
4. Triển khai trên EC2 gồm những container nào, cổng nào mở ra ngoài?